# S11-18 — Feature Importance CWRU — Scénario by_severity

| Champ | Valeur |
|-------|--------|
| **Expériences** | exp_101 (KMeans), exp_104 (Mahalanobis), exp_107 (EWC), exp_110 (HDC) |
| **Scénario** | by_severity : 0.007" → 0.014" → 0.021" |
| **Features** | 9 features temporelles (max, min, mean, sd, rms, skewness, kurtosis, crest, form) |
| **Sprint** | Sprint 11 — S11-18 |


## Section 1 — Setup, imports et chargement des résultats

In [1]:
import json
import os
import sys
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

_cwd = Path('.').resolve()
if _cwd.name == 'cwru_feature_importance':
    os.chdir(_cwd.parent.parent.parent)
elif _cwd.name == 'cl_eval':
    os.chdir(_cwd.parent.parent)

sys.path.insert(0, str(Path('.').resolve()))

from src.evaluation.feature_importance import (
    FEATURE_NAMES_CWRU,
    plot_feature_importance,
    plot_feature_importance_comparison,
)

FIGURES_DIR = Path("notebooks/figures/cl_evaluation/cwru_feature_importance")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

EXPS = {
    "KMeans":      "experiments/exp_101/results/feature_importance.json",
    "Mahalanobis": "experiments/exp_104/results/feature_importance.json",
    "EWC":         "experiments/exp_107/results/feature_importance.json",
    "HDC":         "experiments/exp_110/results/feature_importance.json",
}

data = {model: json.load(open(path)) for model, path in EXPS.items()}
TASKS = ["007", "014", "021"]
print("Chargement OK — modèles :", list(data.keys()))
print("Features :", FEATURE_NAMES_CWRU)


Chargement OK — modèles : ['KMeans', 'Mahalanobis', 'EWC', 'HDC']
Features : ['max', 'min', 'mean', 'sd', 'rms', 'skewness', 'kurtosis', 'crest', 'form']


## Section 2 — Tableau récapitulatif global (ranking par modèle)

In [2]:
rows = {}
for model, d in data.items():
    scores = d["global"]["permutation_importance"]
    rows[model] = {f: scores.get(f, 0.0) for f in FEATURE_NAMES_CWRU}

df = pd.DataFrame(rows).T
df_rank = df.rank(axis=1, ascending=False).astype(int)

print("=== Scores d'importance (permutation globale) ===")
display(df.round(4))
print()
print("=== Rang par feature (1 = plus importante) ===")
display(df_rank)


=== Scores d'importance (permutation globale) ===


,max,min,mean,sd,rms,skewness,kurtosis,crest,form
KMeans,-0.0251,-0.0247,0.0701,-0.0143,-0.0113,-0.0208,-0.0632,-0.0433,-0.0087
Mahalanobis,-0.4675,-0.4654,-0.0983,-0.6667,-0.6636,-0.0532,-0.2476,-0.1939,-0.2827
EWC,0.0069,-0.0004,0.0165,0.0043,0.0000,0.0909,0.0290,0.0333,0.0152
HDC,0.0264,0.0126,-0.0039,0.0589,0.0320,0.0104,0.0000,0.0147,0.0035



=== Rang par feature (1 = plus importante) ===


,max,min,mean,sd,rms,skewness,kurtosis,crest,form
KMeans,7,6,1,4,3,5,9,8,2
Mahalanobis,7,6,2,9,8,1,4,3,5
EWC,6,9,4,7,8,1,3,2,5
HDC,3,5,9,1,2,6,8,4,7


## Section 3 — Barplot groupé global (4 modèles × 9 features)

In [3]:
importances_dict = {
    model: d["global"]["permutation_importance"]
    for model, d in data.items()
}

fig = plot_feature_importance_comparison(
    importances_dict,
    FEATURE_NAMES_CWRU,
    title="Importance globale — CWRU by_severity (4 modèles)",
)
fig.savefig(FIGURES_DIR / "by_severity_global_comparison.png", bbox_inches="tight")
plt.show()
print(f"Figure sauvegardée : {FIGURES_DIR / 'by_severity_global_comparison.png'}")


Figure sauvegardée : notebooks/figures/cl_evaluation/cwru_feature_importance/by_severity_global_comparison.png


/tmp/ipykernel_95738/3552011123.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 4 — Heatmap per-task (3 sévérités × 9 features) par modèle

In [4]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle("Importance par sévérité — CWRU by_severity", fontsize=13, fontweight="bold")

for ax, (model, d) in zip(axes, data.items()):
    mat = np.array([
        [d["per_task"][t]["permutation_importance"].get(f, 0.0) for f in FEATURE_NAMES_CWRU]
        for t in TASKS
    ])
    vmax = max(abs(mat).max(), 1e-6)
    im = ax.imshow(mat, aspect="auto", cmap="RdYlGn", vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(len(FEATURE_NAMES_CWRU)))
    ax.set_xticklabels(FEATURE_NAMES_CWRU, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(TASKS)))
    ax.set_yticklabels([f'sev. {t}"' for t in TASKS], fontsize=9)
    ax.set_title(model, fontsize=11, fontweight="bold")
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "by_severity_per_task_heatmap.png", bbox_inches="tight")
plt.show()
print(f"Figure sauvegardée : {FIGURES_DIR / 'by_severity_per_task_heatmap.png'}")


Figure sauvegardée : notebooks/figures/cl_evaluation/cwru_feature_importance/by_severity_per_task_heatmap.png


/tmp/ipykernel_95738/3573531974.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 5 — EWC : permutation vs gradient saliency (normalisés)

In [5]:
def minmax(d):
    vals = np.array(list(d.values()))
    mn, mx = vals.min(), vals.max()
    if mx == mn:
        return {k: 0.0 for k in d}
    return {k: (v - mn) / (mx - mn) for k, v in d.items()}

ewc = data["EWC"]["global"]
perm_n = minmax(ewc["permutation_importance"])
grad_n = minmax(ewc["gradient_saliency"])

importances_ewc = {"Permutation (EWC)": perm_n, "Gradient Saliency (EWC)": grad_n}
fig = plot_feature_importance_comparison(
    importances_ewc,
    FEATURE_NAMES_CWRU,
    title="EWC — Permutation vs Gradient Saliency (normalisés) — CWRU by_severity",
)
plt.show()

from scipy.stats import spearmanr
rho, p = spearmanr(
    [perm_n[f] for f in FEATURE_NAMES_CWRU],
    [grad_n[f] for f in FEATURE_NAMES_CWRU],
)
print(f"Corrélation Spearman : ρ = {rho:.3f}  (p = {p:.3f})")


Corrélation Spearman : ρ = 0.817  (p = 0.007)


/tmp/ipykernel_95738/1639175924.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 6 — HDC : permutation vs feature masking

In [6]:
hdc = data["HDC"]["global"]
perm_h_n = minmax(hdc["permutation_importance"])
mask_h_n = minmax(hdc["feature_masking_importance"])

importances_hdc = {"Permutation (HDC)": perm_h_n, "Masking (HDC)": mask_h_n}
fig = plot_feature_importance_comparison(
    importances_hdc,
    FEATURE_NAMES_CWRU,
    title="HDC — Permutation vs Feature Masking (normalisés) — CWRU by_severity",
)
plt.show()

from scipy.stats import spearmanr
rho, p = spearmanr(
    [perm_h_n[f] for f in FEATURE_NAMES_CWRU],
    [mask_h_n[f] for f in FEATURE_NAMES_CWRU],
)
print(f"Corrélation Spearman Permutation/Masking (HDC) : ρ = {rho:.3f}  (p = {p:.3f})")


Corrélation Spearman Permutation/Masking (HDC) : ρ = -0.100  (p = 0.798)


/tmp/ipykernel_95738/3188440571.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 7 — Analyse : CWRU by_severity

### Questions clés

**Les features importantes changent-elles quand le défaut s'aggrave ?**

Observer la heatmap (Section 4) par rangée (sévérité) :
- Si `crest_factor` ou `peak` (ici `crest` et `max`) montent en importance à sévérité 0.021" → cohérence physique attendue
- Si le profil est identique pour les 3 sévérités → pas de gradient d'importance lié à l'amplitude du défaut

**`crest` et `max` augmentent-ils avec la sévérité ?**

- En physique des défauts de roulement : les chocs deviennent plus amples avec la taille du défaut
- `crest = max/rms` capte exactement ce ratio → attendu en progression croissante

### Comparaison avec by_fault_type

| Observation | by_fault_type | by_severity |
|-------------|---------------|-------------|
| Top feature globale | À lire Section 2 | À lire Section 2 |
| Stabilité inter-tâche | À lire heatmap | À lire heatmap |
| Cohérence physique | Kurtosis → choc | Crest/Peak → amplitude |

### Notes pour le manuscrit

- `FIXME(gap1)` : progression de `crest` avec sévérité → argument validation physique
- Features recommandées MCU : **kurtosis + crest** (2 features suffisantes ?)
- Comparer avec `by_fault_type` pour identifier des invariants
